# sep07 — leaked seeds 1–2 at step 50, one paired session (Kaggle T4 x2)

**Settings → Accelerator → GPU T4 x2, Internet → On.** Secrets: `HF_TOKEN`.
No training — six arms, one vLLM session, greedy, test split, generations captured.
Budget ~4 GPU-hours. Resumable: rerun skips anything already evaluated.

**Why**: the paper's "the RL stage contributes as much as SFT" rests on seed 0 in one
session (26→52→80→4). This run makes the improves-half n=3 and captures the leaked
seeds' generations, which never existed (their aug sessions kept records only) —
yielding their step-50 AND endpoint copy rates in the bargain.

**Arms** (leaked-run seeds; adapters from the private ckpt repos):
base, sft (clean), leaked s1 ckpt-50, leaked s2 ckpt-50, leaked s1 final, leaked s2 final.

**Prespecified expectations, written before the run**: base reproduces 26/648 exactly
(converged, greedy); leaked finals reproduce 7 and 11 exactly (deterministic copiers);
SFT decodes 52–56; seed 0's step 50 decoded 77–80 across sessions. The claim under
test: each leaked seed's step-50 gain over SFT is positive and comparable to the
SFT-over-base gain. If either seed's step 50 lands at or below SFT, the abstract's
"+0.161/+0.173" story is seed-0-specific — report either way. Expected final copy
rates ~0.9 (seed 0: 0.914/0.924); step-50 copy expected low.


In [ ]:
# Cell 1 — setup (no training: no trl pin needed; vllm installed directly).
import os, json, subprocess, time
T0 = time.time()
def elapsed(): print(f'[budget] {(time.time()-T0)/3600:.2f}h elapsed')
from kaggle_secrets import UserSecretsClient
S = UserSecretsClient()
os.environ['HF_TOKEN'] = S.get_secret('HF_TOKEN')
HF_USER = 'jacksonlukas'

!git clone -b main https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip install -q -e . openai peft accelerate
r = subprocess.run(['python','-c','import connections_rl; print("import OK")'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr); assert r.returncode == 0
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!make data
r = subprocess.run(['python','-c',(
    'from connections_rl.data.loader import load_puzzles;'
    'print(len(load_puzzles("data/splits/puzzles_test.json")))')], capture_output=True, text=True)
assert r.stdout.strip() == '162', 'test split is not 162'
# reclaim any prior version's outputs so reruns resume
import glob, shutil
for src in glob.glob('/kaggle/input/*/results-analysis/sep07'):
    shutil.copytree(src, 'results-analysis/sep07', dirs_exist_ok=True); print('reclaimed', src)
os.makedirs('results-analysis/sep07', exist_ok=True)
print('setup OK'); elapsed()


In [ ]:
# Cell 2 — adapters from the Hub: SFT init + both leaked seeds' ckpt-50 and final.
from huggingface_hub import snapshot_download
tok = os.environ['HF_TOKEN']
if not os.path.exists('artifacts/sft-7b/adapter_config.json'):
    snapshot_download(f'{HF_USER}/connections-rl-sft-7b', local_dir='artifacts/sft-7b', token=tok)
for s in [1, 2]:
    repo = f'{HF_USER}/connections-rl-grpo-7b-seed{s}-ckpt'
    for step in [50, 403]:
        dst = f'artifacts/leaked-s{s}'
        if not os.path.exists(f'{dst}/checkpoint-{step}/adapter_config.json'):
            snapshot_download(repo, local_dir=dst, allow_patterns=f'checkpoint-{step}/*', token=tok)
        assert os.path.exists(f'{dst}/checkpoint-{step}/adapter_config.json'), (s, step)
print('adapters OK'); elapsed()


In [ ]:
# Cell 3 — the session: six arms, one vLLM serve, greedy, generations captured.
!pip install -q vllm
!pip show vllm | grep -E '^(Name|Version)' | tee results-analysis/sep07/session-versions.txt
import urllib.request
def serve(mods):
    proc = subprocess.Popen(
        'vllm serve Qwen/Qwen2.5-7B-Instruct --dtype half --tensor-parallel-size 2 '
        '--enable-lora --enforce-eager --max-lora-rank 16 --max-model-len 2048 '
        '--gpu-memory-utilization 0.85 --lora-modules ' + ' '.join(mods),
        shell=True, stdout=open('/kaggle/working/vllm.log','a'), stderr=subprocess.STDOUT)
    for _ in range(150):
        try: urllib.request.urlopen('http://localhost:8000/health'); print('vLLM ready'); return proc
        except Exception: time.sleep(10)
    raise RuntimeError('vLLM failed -- see /kaggle/working/vllm.log')

ARMS = {'base': 'Qwen/Qwen2.5-7B-Instruct', 'sft': 'connections-rl-sft-7b',
        'leaked-s1-ckpt50': 'leaked-s1-ckpt50', 'leaked-s2-ckpt50': 'leaked-s2-ckpt50',
        'leaked-s1-final': 'leaked-s1-final', 'leaked-s2-final': 'leaked-s2-final'}
out_dir = 'results-analysis/sep07/step50-session-test'
if not os.path.exists(out_dir + '/leaked-s2-final/metrics.json'):
    mods = ['connections-rl-sft-7b=artifacts/sft-7b',
            'leaked-s1-ckpt50=artifacts/leaked-s1/checkpoint-50',
            'leaked-s2-ckpt50=artifacts/leaked-s2/checkpoint-50',
            'leaked-s1-final=artifacts/leaked-s1/checkpoint-403',
            'leaked-s2-final=artifacts/leaked-s2/checkpoint-403']
    proc = serve(mods)
    lines = ['puzzles: data/splits/puzzles_test.json', f'out_dir: {out_dir}',
             'n_resamples: 1000', 'seed: 0', 'capture_generations: true', '', 'arms:']
    for a, mname in ARMS.items():
        lines += [f'  - name: {a}', f'    model: {mname}', '    temperature: 0.0']
    open('results-analysis/sep07/step50_test.yaml','w').write('\n'.join(lines) + '\n')
    r = subprocess.run(['python','-m','connections_rl.eval.run',
                        '--config','results-analysis/sep07/step50_test.yaml'])
    assert r.returncode == 0, 'eval failed'
    !pkill -f 'vllm serve' 2>/dev/null || true
print('session done'); elapsed()


In [ ]:
# Cell 4 — copy rates (published parser: split(','), 16-word guard) + paired contrasts.
import re as _re, random
def parse_groups(text):
    m = _re.search(r'<ANSWER>(.*?)</ANSWER>', text, _re.S)
    if not m: return None
    gs = [[w.strip().upper() for w in gm.group(1).split(',')]
          for line in m.group(1).strip().splitlines()
          if (gm := _re.match(r'\s*Group \d+:\s*(.+)', line))]
    return gs if len(gs) == 4 else None
def prompt_words(pf):
    chat = json.loads(pf) if isinstance(pf, str) else pf
    u = next(m['content'] for m in chat if m['role'] == 'user')
    return [w.strip().upper() for w in u.replace('Words:', '', 1).split(',')]
def copy_stats(arm):
    d = f'{out_dir}/{arm}'
    m = json.load(open(f'{d}/metrics.json')); o = m['summary']['OVERALL']
    quad = tot = pure = parsed = 0
    for line in open(f'{d}/generations.jsonl'):
        r = json.loads(line)
        w = prompt_words(r['prompt']); gs = parse_groups(r['generation'])
        if gs is None or len(w) != 16: continue
        parsed += 1
        quads = [set(w[i:i+4]) for i in range(0, 16, 4)]
        h = sum(1 for g in gs if set(g) in quads)
        quad += h; tot += 4; pure += (h == 4)
    return {'groups_mean_ci': o['groups_correct'], 'slots': round(o['groups_correct'][0]*162),
            'invalid_ci': o['invalid_rate'], 'reward_ci': o['reward'],
            'copy_group_rate': quad/tot if tot else None,
            'pure_copy_rate': pure/parsed if parsed else None, 'parsed': parsed}
def per_puzzle(arm):
    return {json.loads(l)['puzzle_id']: json.loads(l)['groups_correct']
            for l in open(f'{out_dir}/{arm}/records.jsonl')}
def paired(a, b, B=1000):  # mean(a-b) on 0-4 scale, percentile bootstrap over puzzles
    pa, pb = per_puzzle(a), per_puzzle(b)
    ids = sorted(set(pa) & set(pb)); d = [pa[i]-pb[i] for i in ids]
    rng = random.Random(0); n = len(d)
    bs = sorted(sum(d[rng.randrange(n)] for _ in range(n))/n for _ in range(B))
    return {'mean': sum(d)/n, 'lo': bs[int(0.025*B)], 'hi': bs[int(0.975*B)], 'n': n}

summary = {'session': 'sep07 step-50 session (leaked seeds 1-2)', 'decoding': 'greedy T=0.0',
           'arms': {}, 'paired_contrasts': {}}
hdr = f"{'arm':<18} {'groups(0-4)':<24} {'slots':<7} {'invalid':<9} copy g/resp"
print(hdr); print('-'*len(hdr))
for arm in ARMS:
    row = copy_stats(arm); summary['arms'][arm] = row
    g = row['groups_mean_ci']
    print(f"{arm:<18} {g[0]:.4f} [{g[1]:.3f},{g[2]:.3f}]   {row['slots']:<7} "
          f"{row['invalid_ci'][0]:<9.3f} {row['copy_group_rate']:.3f}/{row['pure_copy_rate']:.3f}")
print()
for s in [1, 2]:
    for name, a, b in [(f's{s}: ckpt50-sft', f'leaked-s{s}-ckpt50', 'sft'),
                       (f's{s}: ckpt50-base', f'leaked-s{s}-ckpt50', 'base'),
                       (f's{s}: final-ckpt50', f'leaked-s{s}-final', f'leaked-s{s}-ckpt50'),
                       ('sft-base' if s == 1 else None, 'sft', 'base')]:
        if name is None: continue
        c = paired(a, b); summary['paired_contrasts'][name] = c
        print(f"{name:<18} {c['mean']:+.4f} [{c['lo']:+.4f},{c['hi']:+.4f}]  n={c['n']}")
print()
# anchors that must reproduce exactly; loud warning if not
b = summary['arms']['base']['slots']
print(f"anchor check: base {b}/648 (expect exactly 26)")
for s, expect in [(1, 7), (2, 11)]:
    got = summary['arms'][f'leaked-s{s}-final']['slots']
    print(f"anchor check: leaked-s{s}-final {got}/648 (expect exactly {expect})"
          + ('' if got == expect else '  <-- DOES NOT REPRODUCE, investigate before using'))
# the claim under test
sft_gain = summary['paired_contrasts']['sft-base']['mean']
for s in [1, 2]:
    g = summary['paired_contrasts'][f's{s}: ckpt50-sft']
    verdict = 'positive' if g['lo'] > 0 else ('NOT resolved above SFT' if g['hi'] > 0 else 'BELOW SFT')
    print(f"seed {s}: step-50 gain over SFT {g['mean']:+.4f} [{g['lo']:+.4f},{g['hi']:+.4f}] -> {verdict}"
          f" (SFT-over-base in this session: {sft_gain:+.4f})")
json.dump(summary, open('results-analysis/sep07/step50_summary.json','w'), indent=1)
print('wrote results-analysis/sep07/step50_summary.json')


In [ ]:
# Cell 5 — persist.
!zip -qr /kaggle/working/sep07-step50-outputs.zip results-analysis/sep07
print('zip ready: /kaggle/working/sep07-step50-outputs.zip')
try:
    from huggingface_hub import HfApi
    HfApi(token=os.environ['HF_TOKEN']).upload_folder(
        folder_path='results-analysis/sep07',
        repo_id=f'{HF_USER}/connections-rl-results', repo_type='dataset',
        path_in_repo='sep07')
    print('Hub upload OK -> connections-rl-results/sep07')
except Exception as e:
    print('Hub upload failed (fine -- use the zip):', e)
elapsed()
